> ⚠️ **机器翻译提示**：本 notebook 由机器翻译生成，未经人工校对，如发现翻译错误敬请谅解（代码与计算结果和越南语原版完全一致，只有文字说明是机器翻译）。

# 第04单元 — 分布、统计显著性、效应量、检验力
第04单元专属 notebook —— 重现：离群值清理、95%置信区间、偏度与Shapiro-Wilk检验、Cohen's d、eta²、基于检验力的样本量表。

SPSS/PSPP 对应操作：`Transform > Compute`、`DESCRIPTIVES`、`EXAMINE`、`T-TEST`、`ONEWAY`。

In [ ]:
# 先运行这一格（Colab：点击 ▶）。无需安装任何库。
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """模拟数据集"240人班级" —— 第03-08单元统一使用 seed=2026，
    与仓库中的 lop_hoc_240.csv/.sav 完全一致（非真实数据，但100%可复现）。"""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["男", "女"], n)
    method = rng.choice(["传统教学", "项目式教学"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "项目式教学")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "项目式教学") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("已生成 df：", df.shape)

## 0. 首先清理 `study_hours` 的离群值
一名学生的 `study_hours=48` —— 几乎可以肯定是把「4.8」误输入为「48」（见第01单元）。若跳过此步骤，偏度会得到约7.4而非0.658。

In [ ]:
mask = df["study_hours"] > 20
print("被修正的行数：", mask.sum(), "| 修正前的值：", df.loc[mask, "study_hours"].tolist())
df.loc[mask, "study_hours"] = df.loc[mask, "study_hours"] / 10
print("修正后的值：", df.loc[mask, "study_hours"].tolist())

## 1. 数学平均分的95%置信区间
公式：CI₉₅ = x̄ ± 1.96 × SE，其中 SE = s/√n。

🎯 **这个方法用来回答什么问题？** 置信区间（CI）回答：如果多次重复抽样研究，哪个取值范围有可能包含总体的真实参数。与只报告一个样本平均值不同，CI 还告诉你这个数字伴随的不确定程度——样本越小，CI 越宽（越不确定）。

In [ ]:
m = df["math"].mean(); s = df["math"].std(ddof=1); n = len(df)
se = s / np.sqrt(n)
ci_low, ci_high = m - 1.96*se, m + 1.96*se
print(f"平均值={m:.2f}  标准差={s:.2f}  标准误={se:.3f}")
print(f"95%置信区间 = [{ci_low:.2f}; {ci_high:.2f}]")

#### 📤 实际输出
平均值=47.91，标准误=0.625 → 95%置信区间=[46.68; 49.14]。✅ 与课程数字一致。

## 2. 偏度与Shapiro-Wilk检验：`study_hours`（已清理）与 `pretest`
经验法则：将偏度与 ±2×SE(偏度) 比较，SE(偏度) ≈ √(6/n)。

🎯 **这个方法用来回答什么问题？** 偏度/Shapiro-Wilk检验回答：数据是否足够接近正态分布，可以使用参数检验（t检验、方差分析——它们都假设数据服从正态分布）？这一步是在检查方法的适用前提，而不是研究结果本身——如果严重违反正态性，应改用非参数检验（第07单元）。

In [ ]:
se_skew = np.sqrt(6/n)
for col in ["study_hours", "pretest"]:
    sk = stats.skew(df[col], bias=False)
    w, p = stats.shapiro(df[col])
    verdict = "显著偏斜" if abs(sk) > 2*se_skew else "在可接受范围内"
    print(f"{col}：偏度={sk:+.3f}（阈值±{2*se_skew:.3f}，{verdict}）| Shapiro-Wilk W={w:.3f}, p={p:.4f}")

#### 📤 实际输出
`study_hours`：偏度=+0.658，阈值±0.314 → 超出阈值，Shapiro-Wilk p<0.001（不支持「数据服从正态分布」的假设）。`pretest`：偏度=-0.097，在范围内，Shapiro-Wilk p=0.117（没有足够证据拒绝正态性）。✅ 与课程一致。

## 3. Cohen's d —— 不同教学方法的成绩提升幅度

🎯 **这个方法用来回答什么问题？** Cohen's d 回答：两组之间的差异有多大，以「标准差的个数」为单位——它补充了p值（p值只说明是否具有统计显著性）无法回答的问题：这个差异在实际意义上究竟有多大。

In [ ]:
df["gain"] = df["posttest"] - df["pretest"]
g1 = df.loc[df["method"]=="项目式教学", "gain"]
g2 = df.loc[df["method"]=="传统教学", "gain"]
sp = np.sqrt(((len(g1)-1)*g1.var(ddof=1) + (len(g2)-1)*g2.var(ddof=1)) / (len(g1)+len(g2)-2))
d = (g1.mean() - g2.mean()) / sp
t, p = stats.ttest_ind(g1, g2)
print(f"项目式教学：n={len(g1)}, M={g1.mean():.2f}, SD={g1.std(ddof=1):.2f}")
print(f"传统教学：n={len(g2)}, M={g2.mean():.2f}, SD={g2.std(ddof=1):.2f}")
print(f"t={t:.3f}, p={p:.2e}")
print(f"Cohen's d = {d:.3f}")

#### 📤 实际输出
d≈0.981（中到大效应，接近「大」效应阈值1.00），t=7.589（p<0.001）。✅ 与课程一致。

## 4. Eta² —— 学校 → 数学成绩（与第06单元的方差分析配合使用）

🎯 **这个方法用来回答什么问题？** Eta²回答：因变量的变异中有百分之多少可以被分组变量「解释」——类似回归中的R²，这是专门用于方差分析的一种效应量，用来补充p值（p值只说明是否显著，不说明效应有多大）。

In [ ]:
grand = df["math"].mean()
ssb = sum(len(g)*(g["math"].mean()-grand)**2 for _, g in df.groupby("school"))
sst = ((df["math"]-grand)**2).sum()
eta2 = ssb / sst
print(f"组间平方和={ssb:.2f}  总平方和={sst:.2f}  eta²={eta2:.4f}")

#### 📤 实际输出
eta²≈0.047 → 效应量很小，尽管F检验具有统计显著性（p=0.003）！这正好证明统计显著性和效应量是两个不同的问题。

## 5. 基于检验力的样本量表（Cohen, 1988, statsmodels）
如尚未安装 `statsmodels` 请先安装：Colab 上已预装。

🎯 **这个方法用来回答什么问题？** 检验力分析回答：需要多少观测值，才有合理的机会发现一个真实存在的效应（如果它确实存在）——这是在收集数据之前就要做的事，而不是事后。如果样本量太小，即使真实存在的效应也很容易被漏掉（第二类错误）。

In [ ]:
from statsmodels.stats.power import TTestIndPower
analysis = TTestIndPower()
rows = []
for d_val, label in [(0.2,"小"), (0.5,"中"), (0.8,"大")]:
    n_req = analysis.solve_power(effect_size=d_val, alpha=0.05, power=0.80, alternative="two-sided")
    rows.append((d_val, label, round(n_req)))
pd.DataFrame(rows, columns=["效应量 d", "等级", "每组样本量"])

#### 📤 实际输出
d=0.2→约394/组，d=0.5→约64/组，d=0.8→约26/组（α=0.05，power=0.80）。✅ 与第02、04单元中的表格一致。